In [1]:
import sys
import numpy as np
from PyQt6.QtWidgets import (QApplication, QMainWindow, QWidget, QVBoxLayout, 
                             QHBoxLayout, QLabel, QPushButton, QSlider)
from PyQt6.QtCore import QTimer, Qt
import pyqtgraph as pg

class MRA_Octave_Fixed_Final(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("RL-LQR: Implementación Octave (Fix Matmul)")
        
        # --- Parámetros Octave ---
        self.A = np.array([[1.8980, -0.9048], [1.0, 0.0]])
        self.B = np.array([[1.0], [0.0]])
        self.Q = np.eye(2)
        self.R = 2.0
        self.gamma = 0.999
        self.f_olvido = 0.99
        self.con_error_k = 10e-2
        self.N = 4000
        self.n_ls = 7
        
        # H y K inicial de Octave
        self.H_init = np.array([[14.0, -2.0, 2.0], [-8.0, 3.0, -1.0], [8.0, -5.0, 4.0]])
        self.K = -(1.0 / self.H_init[2, 2]) * self.H_init[2, 0:2]
        self.K = self.K.reshape(1, 2)
        
        self.x = np.array([[5.0], [-4.0]])
        self.W_H = np.zeros((6, 1))
        self.P_n = np.eye(6) * 1000
        self.phi_ls = np.zeros((6, self.n_ls))
        self.phi1_ls = np.zeros((6, self.n_ls))
        self.r_ls = np.zeros((self.n_ls, 1))
        
        self.iter = 1
        self.pos_history = []
        
        self.init_ui()
        self.timer = QTimer()
        self.timer.timeout.connect(self.step)
        self.timer.start(20)

    def init_ui(self):
        central_widget = QWidget()
        self.setCentralWidget(central_widget)
        layout = QHBoxLayout(central_widget)

        self.canvas = pg.GraphicsLayoutWidget()
        self.view = self.canvas.addViewBox()
        self.view.setRange(xRange=[-10, 10], yRange=[-10, 10])
        self.rect = pg.ScatterPlotItem(size=30, brush='r', symbol='s')
        self.view.addItem(self.rect)
        layout.addWidget(self.canvas, stretch=2)

        right_panel = QVBoxLayout()
        self.plot_res = pg.PlotWidget(title="Respuesta Temporal")
        self.curve = self.plot_res.plot(pen='y')
        right_panel.addWidget(self.plot_res)
        
        self.label_info = QLabel("Iteración: 0")
        self.label_err = QLabel("Error TD: 0.0000")
        self.label_k = QLabel("K: [0, 0]")
        right_panel.addWidget(self.label_info)
        right_panel.addWidget(self.label_err)
        right_panel.addWidget(self.label_k)

        self.slider_k1 = QSlider(Qt.Orientation.Horizontal)
        self.slider_k2 = QSlider(Qt.Orientation.Horizontal)
        for s in [self.slider_k1, self.slider_k2]:
            s.setRange(-200, 200)
            s.setEnabled(False)
            right_panel.addWidget(s)

        layout.addLayout(right_panel, stretch=1)

    def step(self):
        if self.iter > self.N:
            self.timer.stop()
            return

        # Ruido y Control
        noise = 0.20099 * np.exp(-0.000269 * self.iter) * (np.sin(self.iter)**3 * np.cos(self.iter))
        u = float(np.dot(self.K, self.x)) + noise
        
        x_curr = self.x
        x_next = np.dot(self.A, x_curr) + self.B * u
        u2 = float(np.dot(self.K, x_next))
        r_val = float(np.dot(np.dot(x_curr.T, self.Q), x_curr) + (u**2) * self.R)

        e_d_t_val = 0.0

        if self.iter <= self.n_ls:
            idx = self.iter - 1
            self.r_ls[idx, 0] = r_val
            self.phi_ls[:, idx] = [x_curr[0,0]**2, x_curr[0,0]*x_curr[1,0], x_curr[0,0]*u, 
                                   x_curr[1,0]**2, x_curr[1,0]*u, u**2]
            self.phi1_ls[:, idx] = [x_next[0,0]**2, x_next[0,0]*x_next[1,0], x_next[0,0]*u2, 
                                    x_next[1,0]**2, x_next[1,0]*u2, u2**2]
            
            if self.iter == self.n_ls:
                diff_phi = self.phi_ls - self.phi1_ls
                self.W_H = np.dot(np.linalg.pinv(np.dot(diff_phi, diff_phi.T)), np.dot(diff_phi, self.r_ls))
        
        else:
            phi = np.array([x_curr[0,0]**2, x_curr[0,0]*x_curr[1,0], x_curr[0,0]*u, 
                            x_curr[1,0]**2, x_curr[1,0]*u, u**2]).reshape(-1, 1)
            phi1 = np.array([x_next[0,0]**2, x_next[0,0]*x_next[1,0], x_next[0,0]*u2, 
                             x_next[1,0]**2, x_next[1,0]*u2, u2**2]).reshape(-1, 1)
            
            vec_act = phi - self.gamma * phi1
            e_d_t_val = r_val - float(np.dot(self.W_H.T, vec_act))
            
            # --- CÁLCULO DE RLS CON NP.DOT PARA EVITAR EL ERROR DE DIMENSIONES ---
            a_inv = 1.0 / (1.0 - self.f_olvido)
            p_n_scaled = (1.0 / self.f_olvido) * self.P_n
            
            # Denominador: a_inv + vec_act' * P_scaled * vec_act
            tmp_den = np.dot(np.dot(vec_act.T, p_n_scaled), vec_act)
            den = a_inv + float(tmp_den)
            
            # L = P_scaled * vec_act / den
            L = np.dot(p_n_scaled, vec_act) / den
            
            # Actualización de W_H y P_n
            self.W_H = self.W_H + L * e_d_t_val
            self.P_n = np.dot((np.eye(6) - np.dot(L, vec_act.T)), p_n_scaled)
            
            # Condicional de error para actualizar K
            if abs(e_d_t_val) < self.con_error_k:
                W = self.W_H.flatten()
                H_new = np.array([
                    [W[0],   W[1]/2.0, W[2]/2.0],
                    [W[1]/2.0, W[3],   W[4]/2.0],
                    [W[2]/2.0, W[4]/2.0, W[5]]
                ])
                if abs(H_new[2, 2]) > 1e-4:
                    self.K = -(1.0 / H_new[2, 2]) * H_new[2, 0:2]
                    self.K = self.K.reshape(1, 2)
                    self.slider_k1.setValue(int(self.K[0,0] * 10))
                    self.slider_k2.setValue(int(self.K[0,1] * 10))

        self.x = x_next
        self.iter += 1
        
        self.rect.setData(x=[0], y=[float(self.x[0,0])])
        self.pos_history.append(float(self.x[0,0]))
        if len(self.pos_history) > 400: self.pos_history.pop(0)
        self.curve.setData(self.pos_history)
        self.label_info.setText(f"Iteración: {self.iter} / {self.N}")
        self.label_err.setText(f"Error TD: {abs(e_d_t_val):.6f}")
        self.label_k.setText(f"K1: {self.K[0,0]:.4f} K2: {self.K[0,1]:.4f}")

if __name__ == "__main__":
    app = QApplication(sys.argv)
    sim = MRA_Octave_Fixed_Final()
    sim.show()
    sys.exit(app.exec())

/tmp/ipykernel_16683/459656247.py:84: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  u = float(np.dot(self.K, self.x)) + noise
/tmp/ipykernel_16683/459656247.py:88: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  u2 = float(np.dot(self.K, x_next))
/tmp/ipykernel_16683/459656247.py:89: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  r_val = float(np.dot(np.dot(x_curr.T, self.Q), x_curr) + (u**2) * self.R)
/tmp/ipykernel_16683/459656247.py:84: DeprecationWarning: Conversi

SystemExit: 0

/home/jm-liberty/miniconda3/envs/rl_test/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
